# NetSentry — Data Cleaning and Classification

**Goal:** Predict Allow, Deny, Drop or Reset. These are **categories**, so this is classification, not linear regression.

We follow the professor's workflow: import, inspect, clean, choose X/y, split, build pipelines, cross-validate, and evaluate.
The original UCI dataset has 65,532 records.

## 1. Import the CSV

This uses the same \`pd.read_csv()\` method shown in class. In Colab upload \`log2.csv\` using the Files panel.
The notebook also works from the project root or its \`notebooks\` directory.

In [ ]:
from pathlib import Path
import pandas as pd

if Path("log2.csv").exists():            # Google Colab upload
    csv_file = "log2.csv"
elif Path("data/log2.csv").exists():     # Project root
    csv_file = "data/log2.csv"
else:                                   # notebooks directory
    csv_file = "../data/log2.csv"

df = pd.read_csv(csv_file)
print("Rows and columns:", df.shape)
df.head()

## 2. Inspect the dataset

Look at the rows, data types, missing values and duplicate records before cleaning.

In [ ]:
df.info()
display(df.describe())
print("Missing values:", df.isna().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Action counts:", df["Action"].value_counts())

## 3. Clean the data

Remove exact duplicates, standardize label names, and reject impossible measurements or contradictory labels.
Missing numeric values are left for the training-only \`SimpleImputer\`. We do not remove unusual values just for being large.

In [ ]:
import numpy as np

clean = df.drop_duplicates().copy()
clean["Action"] = clean["Action"].str.lower().str.strip().replace({
    "allow": "Allow", "deny": "Deny",
    "drop": "Drop", "reset-both": "Reset"
})

features = [column for column in clean.columns if column != "Action"]
clean[features] = clean[features].apply(pd.to_numeric, errors="coerce")

# Reject invalid actions, negative or infinite values, and impossible ports.
valid = clean["Action"].isin(["Allow", "Deny", "Drop", "Reset"])
valid &= ~np.isinf(clean[features]).any(axis=1)
valid &= ~clean[features].lt(0).any(axis=1)
valid &= ~clean[features[:4]].gt(65535).any(axis=1)

# Ports, bytes and packet counts must be whole numbers; duration may be decimal.
whole_columns = [column for column in features if column != "Elapsed Time (sec)"]
valid &= ~(clean[whole_columns].notna() & clean[whole_columns].mod(1).ne(0)).any(axis=1)

# Check packet and byte totals wherever those measurements are present.
valid &= ~(clean["Packets"].notna() & clean["Packets"].le(0))
valid &= ~(clean["pkts_sent"] + clean["pkts_received"] > clean["Packets"])
valid &= ~(clean["Bytes Sent"] + clean["Bytes Received"] > clean["Bytes"] * 1.02 + 1e-9)
clean = clean.loc[valid].copy()

# The same measurements must not have two different answers.
label_count = clean.groupby(features, dropna=False)["Action"].transform("nunique")
clean = clean.loc[label_count == 1].copy()

print("Original records:", len(df))
print("Clean records:", len(clean))
print(clean["Action"].value_counts())

## 4. Choose the input features (X) and target (y)

X has 11 numeric network measurements. y is the firewall action.
The inputs are already numeric, so one-hot encoding is unnecessary here.

In [ ]:
X = clean.drop(columns="Action")
y = clean["Action"]

print("Input data:", X.shape)
print("Target:", y.shape)
X.head()

## 5. Split before training

Keep 20% of records untouched for testing. Stratifying preserves each action's proportion.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training:", len(X_train))
print("Testing:", len(X_test))

## 6. Build scikit-learn pipelines

The imputer learns replacement values from training data only.
Logistic Regression and SVM use standard scaling; Random Forest does not require scaling.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1200, class_weight="balanced", random_state=42))
    ]),
    "Linear SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearSVC(class_weight="balanced", random_state=42, max_iter=4000, dual=False))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1))
    ])
}

## 7. Compare models with five-fold cross-validation

Run validation on training records only. Use **macro F1** as well as accuracy because Reset is rare.
The training-validation gap is measured in percentage points.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, candidate in models.items():
    scores = cross_validate(
        candidate, X_train, y_train, cv=folds,
        scoring={"accuracy": "accuracy", "f1": "f1_macro"},
        return_train_score=True, n_jobs=1
    )
    train_acc = scores["train_accuracy"].mean()
    validation_acc = scores["test_accuracy"].mean()
    results.append({
        "Model": name,
        "Validation accuracy": validation_acc,
        "Macro F1": scores["test_f1"].mean(),
        "Accuracy gap (pp)": 100 * (train_acc - validation_acc)
    })

comparison = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(comparison.round(4))

## 8. Final evaluation

NetSentry uses Random Forest on the website. Evaluate on the **untouched** test set.
Report accuracy, per-class performance and the confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = models["Random Forest"]
model.fit(X_train, y_train)
predictions = model.predict(X_test)
actions = ["Allow", "Deny", "Drop", "Reset"]

print("Test accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, labels=actions, zero_division=0))

matrix = confusion_matrix(y_test, predictions, labels=actions)
display(pd.DataFrame(matrix, index=actions, columns=actions))

## Summary

- This is classification, not numerical regression.
- Preprocessing belongs inside training pipelines to prevent data leakage.
- Examine Reset recall and macro F1, not accuracy alone.
- Predictions are based on past firewall records; NetSentry does not control a firewall.

This notebook works standalone in Google Colab after uploading \`log2.csv\`.
For the website, train and save the deployed model with \`train_model.py\`.

**Source:** UCI Internet Firewall Data, DOI 10.24432/C5131M.